In [ ]:
from mpmath import coulombf, coulombg
import numpy as np
import mpmath
import numba
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar
# 设置高精度计算（50位小数）
mpmath.mp.dps = 50
# # 计算 F 和 G
# F_values = [mpmath.coulombf(l, eta, rho) for rho in rho_values]
# G_values = [mpmath.coulombg(l, eta, rho) for rho in rho_values]

In [ ]:
e2=1.43997 ; hbarc=197.3269718 ; amu=931.49432
z1=1 ; m1=1*amu
z2=0 ; m2=1*amu
E=5
mu=m1*m2/(m1+m2)
v0=-72.15 ; a0=1.484 ; r0=0; l=0
k=np.sqrt(2*mu*E/(hbarc**2))
eta= z1 * z2 * e2 * mu / (hbarc**2 * k)

In [ ]:
def gausspot(r, v0, r0, a):
    if a > 1e-6:
        return v0 * np.exp(-(r - r0)**2 / a**2)
    else:
        raise ValueError("a too small in gausspot! a must be > 1e-6")
    
    
    
def vpot(r, v0, r0, a):
    return gausspot(r, v0, r0, a) + hbarc**2 / 2 / mu*l*(l+1)/r**2
    
    
#coulomb function

def F_L(r, k, eta, L):
    """库仑函数 F_L(η, kr)"""
    return coulombf(L, eta, k * r)

def G_L(r, k, eta, L):
    """库仑函数 G_L(η, kr)"""
    return coulombg(L, eta, k * r)

#derivative of F G
#对r的导数，而非对kr的导数
def dF_L(r, k, eta, L, h=1e-8):
    """数值计算 F_L(η, kr) 的导数 (d/dr)"""
    return mpmath.diff(lambda x: mpmath.coulombf(L, eta, k * x), mpmath.mpf(r), n=1, h=h)

def dG_L(r, k, eta, L, h=1e-8):
    """数值计算 G_L(η, kr) 的导数 (d/dr)"""
    return mpmath.diff(lambda x: mpmath.coulombg(L, eta, k * x), mpmath.mpf(r), n=1, h=h)


In [ ]:
# mesh
h = 0.001/k
n_mesh = 60000
r0 = 2*l/h

print("h: ", h)
print("n_mesh: ", n_mesh)
print("r0: ", r0)
print("r0+h*n_mesh: ", r0 + h * n_mesh)
###########
r = np.arange(r0, r0 + h * n_mesh, h)

########
z = np.zeros_like(r, dtype=np.complex128)
y = np.zeros_like(r, dtype=np.complex128)
psi=np.zeros_like(r, dtype=np.complex128)


z[0]=0
z[1]=h*1

f=2*mu/hbarc**2*(vpot(r, v0, r0, a0)-E)

for i in range(1, len(r)-1):
    z[i+1] =2*z[i] - z[i-1] + h**2*f[i]*z[i]
    

y=(1-h**2/12*f)*z
#H,H'
hln=G_L(r[-1], k, eta, l)-1j*F_L(r[-1], k, eta, l)
hlp=G_L(r[-1], k, eta, l)+1j*F_L(r[-1], k, eta, l)
dhln=(dG_L(r[-1], k, eta, l)-1j*dF_L(r[-1], k, eta, l))
dhlp=(dG_L(r[-1], k, eta, l)+1j*dF_L(r[-1], k, eta, l))
#dy
dy=(-y[-1]+8*y[-2]-8*y[-4]+y[-5])/12/h

#S matrix
Sl=(dy*hln-y[-3]*dhln)/(dy*hlp-y[-3]*dhlp)
print("S matrix: ", Sl)

#psi=c*y
c=(hln-Sl*hlp)*(1j)/2/y[-3]
psi=c*y
psi=np.array(psi, dtype=np.complex128)
plt.plot(r, np.real(psi), label='Real part of wave function')
